# Objetivo 2 — Implementacion de Modelos HQCNN


## 0 · Instalacion

In [1]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','torch','torchvision','--index-url','https://download.pytorch.org/whl/cu128'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','pennylane>=0.38','pennylane-lightning','scikit-learn','matplotlib','seaborn','tqdm'],check=False)
import torch; print('PyTorch:',torch.__version__)


PyTorch: 2.11.0+cu128


## 1 · Imports y seed

In [2]:
import os,random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader,WeightedRandomSampler
import pennylane as qml
from pennylane.qnn import TorchLayer
SEED=42
random.seed(SEED);np.random.seed(SEED)
torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PennyLane:',qml.__version__,'| PyTorch:',torch.__version__,'| Device:',DEVICE)


PennyLane: 0.45.0 | PyTorch: 2.11.0+cu128 | Device: cuda


## 2 · DataLoaders

In [3]:
BASE_DIR=Path(os.getcwd())
CHEST_OUT=BASE_DIR/'etl_output'/'chest_xray'
LUNG_OUT=BASE_DIR/'etl_output'/'lung_cancer'
IMG_SIZE=(128,128);MEAN=[0.485,0.456,0.406];STD=[0.229,0.224,0.225]
tfm_tr=T.Compose([T.Resize(IMG_SIZE),T.Grayscale(num_output_channels=3),T.RandomRotation(10),T.RandomHorizontalFlip(),T.ColorJitter(brightness=0.15),T.RandomAffine(degrees=0,scale=(0.90,1.10)),T.ToTensor(),T.Normalize(MEAN,STD)])
tfm_ev=T.Compose([T.Resize(IMG_SIZE),T.Grayscale(num_output_channels=3),T.ToTensor(),T.Normalize(MEAN,STD)])
def make_loaders(root,weighted=False,bs=32):
    root=Path(root)
    ds_tr=ImageFolder(root/'train',transform=tfm_tr)
    ds_va=ImageFolder(root/'val',transform=tfm_ev)
    ds_te=ImageFolder(root/'test',transform=tfm_ev)
    kw=dict(num_workers=2,pin_memory=True)
    if weighted:
        tgts=torch.tensor(ds_tr.targets)
        sw=(1.0/torch.bincount(tgts).float())[tgts]
        ltr=DataLoader(ds_tr,batch_size=bs,sampler=WeightedRandomSampler(sw,len(sw),True),**kw)
    else:
        ltr=DataLoader(ds_tr,batch_size=bs,shuffle=True,**kw)
    return ltr,DataLoader(ds_va,batch_size=bs,shuffle=False,**kw),DataLoader(ds_te,batch_size=bs,shuffle=False,**kw),ds_tr.class_to_idx
print('DataLoaders definidos.')


DataLoaders definidos.


## 3 · Backend PennyLane

In [4]:
def get_qdev(n):
    try: return qml.device('lightning.qubit',wires=n)
    except: return qml.device('default.qubit',wires=n)


## 4 · Modelo 1 — HQC-CNN (Dong et al., 2023)
 backbone CNN liviano + VQC 4 qubits.  
Encoding: **RY** (angle encoding). Entrelazamiento: **circular** (CNOT en anillo).  
Reduccion a 4 features: **Linear entrenable** como aproximacion diferenciable de PCA.  

In [5]:
N_QUBITS_HQCCNN=4
dev_hqccnn=get_qdev(N_QUBITS_HQCCNN)

@qml.qnode(dev_hqccnn,interface='torch',diff_method='adjoint')
def circuit_hqccnn(inputs,weights_rx,weights_ry):
    # Encoding RY segun PPI
    qml.AngleEmbedding(inputs,wires=range(N_QUBITS_HQCCNN),rotation='Y')
    # Capa variacional: RX+RY por qubit + entrelazamiento circular
    n=N_QUBITS_HQCCNN
    for q in range(n):
        qml.RX(weights_rx[q],wires=q)
        qml.RY(weights_ry[q],wires=q)
    for q in range(n): qml.CNOT(wires=[q,(q+1)%n])  # circular
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_HQCCNN)]

WS_HQCCNN={'weights_rx':(N_QUBITS_HQCCNN,),'weights_ry':(N_QUBITS_HQCCNN,)}

class HQCCNN(nn.Module):
    def __init__(self,n_classes):
        super().__init__()
        self.feat=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d((4,4)))
        # Linear entrenable como aproximacion diferenciable de PCA (4 features -> 4 qubits)
        self.pre_q=nn.Sequential(nn.Linear(64*4*4,64),nn.ReLU(),nn.Linear(64,N_QUBITS_HQCCNN),nn.Tanh())
        self.ql=TorchLayer(circuit_hqccnn,WS_HQCCNN)
        self.cls=nn.Linear(N_QUBITS_HQCCNN,n_classes)
    def forward(self,x):
        dv=x.device
        x=self.feat(x).flatten(1)
        qi=self.pre_q(x).float().cpu()*torch.pi
        return self.cls(self.ql(qi).to(dv).to(x.dtype))

with torch.no_grad():
    m=HQCCNN(2)
    print('HQC-CNN output:',m(torch.randn(2,3,128,128)).shape)
    print('Params:',sum(p.numel() for p in m.parameters() if p.requires_grad))


HQC-CNN output: torch.Size([2, 2])
Params: 89686


## 5 · Modelo 2 — PEQML (Abdur & Kim, 2025)
Backbone depthwise-separable + VQC 4 qubits, `BasicEntanglerLayers`, encoding RY.

In [6]:
N_QUBITS_PEQML=4
dev_peqml=get_qdev(N_QUBITS_PEQML)

@qml.qnode(dev_peqml,interface='torch',diff_method='adjoint')
def circuit_peqml(inputs,weights):
    qml.AngleEmbedding(inputs,wires=range(N_QUBITS_PEQML),rotation='Y')
    qml.BasicEntanglerLayers(weights,wires=range(N_QUBITS_PEQML))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_PEQML)]

class PEQML(nn.Module):
    def __init__(self,n_classes):
        super().__init__()
        def dw(ic,oc,s=1): return nn.Sequential(nn.Conv2d(ic,ic,3,stride=s,padding=1,groups=ic,bias=False),nn.Conv2d(ic,oc,1,bias=False),nn.BatchNorm2d(oc),nn.ReLU6())
        self.feat=nn.Sequential(nn.Conv2d(3,8,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(8),nn.ReLU6(),dw(8,16,2),dw(16,32,2),dw(32,32,2),nn.AdaptiveAvgPool2d((2,2)))
        self.pre_q=nn.Sequential(nn.Linear(32*2*2,N_QUBITS_PEQML),nn.Tanh())
        self.ql=TorchLayer(circuit_peqml,{'weights':(2,N_QUBITS_PEQML)})
        self.cls=nn.Linear(N_QUBITS_PEQML,n_classes)
    def forward(self,x):
        dv=x.device
        x=self.feat(x).flatten(1)
        qi=self.pre_q(x).float().cpu()*torch.pi
        return self.cls(self.ql(qi).to(dv).to(x.dtype))

with torch.no_grad():
    m=PEQML(2)
    print('PEQML output:',m(torch.randn(2,3,128,128)).shape)
    print('Params:',sum(p.numel() for p in m.parameters() if p.requires_grad))


PEQML output: torch.Size([2, 2])
Params: 3094


## 6 · Modelo 3 — HQCINN (Akpinar et al., 2025)
**shallow:** 4 qubits, 1 capa | **deep:** **6 qubits**, 3 capas (segun PPI, Akpinar et al.)

In [7]:
N_QUBITS_SHALLOW=4
N_QUBITS_DEEP=6   # segun PPI: deep = 6-8 qubits (Akpinar et al., 2025)

def build_hqcinn_qlayer(n_qubits,n_layers,entanglement='linear'):
    dev=get_qdev(n_qubits)
    def entangle(wires,et):
        n=len(wires)
        if et=='linear':    [qml.CNOT(wires=[wires[i],wires[i+1]]) for i in range(n-1)]
        elif et=='circular':[qml.CNOT(wires=[wires[i],wires[(i+1)%n]]) for i in range(n)]
        elif et=='full':    [qml.CNOT(wires=[wires[i],wires[j]]) for i in range(n) for j in range(i+1,n)]
    @qml.qnode(dev,interface='torch',diff_method='adjoint')
    def circuit(inputs,weights_rx,weights_ry,weights_rz):
        qml.AngleEmbedding(inputs,wires=range(n_qubits),rotation='X')
        for l in range(n_layers):
            for q in range(n_qubits):
                qml.RX(weights_rx[l,q],wires=q)
                qml.RY(weights_ry[l,q],wires=q)
                qml.RZ(weights_rz[l,q],wires=q)
            entangle(list(range(n_qubits)),entanglement)
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
    ws={k:(n_layers,n_qubits) for k in ['weights_rx','weights_ry','weights_rz']}
    return TorchLayer(circuit,ws)

class HQCINN(nn.Module):
    def __init__(self,n_classes,variant='shallow',entanglement='linear'):
        super().__init__()
        n_q,n_l={'shallow':(N_QUBITS_SHALLOW,1),'deep':(N_QUBITS_DEEP,3)}[variant]
        self.n_qubits=n_q
        self.feat=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
        self.pre_q=nn.Sequential(nn.Linear(128*2*2,n_q),nn.Tanh())
        self.ql=build_hqcinn_qlayer(n_q,n_l,entanglement)
        self.cls=nn.Linear(n_q,n_classes)
    def forward(self,x):
        dv=x.device
        x=self.feat(x).flatten(1)
        qi=self.pre_q(x).float().cpu()*torch.pi
        return self.cls(self.ql(qi).to(dv).to(x.dtype))

for v in ['shallow','deep']:
    with torch.no_grad():
        m=HQCINN(2,variant=v)
        p=sum(pp.numel() for pp in m.parameters() if pp.requires_grad)
        print(f'HQCINN-{v}: qubits={m.n_qubits}  out={m(torch.randn(2,3,128,128)).shape}  params={p:,}')


HQCINN-shallow: qubits=4  out=torch.Size([2, 2])  params=95,770
HQCINN-deep: qubits=6  out=torch.Size([2, 2])  params=96,842


## 7 · Factory y tabla de parametros

In [8]:
def build_model(name,n_classes):
    if name=='hqccnn': return HQCCNN(n_classes)
    elif name=='peqml': return PEQML(n_classes)
    elif name=='hqcinn_shallow': return HQCINN(n_classes,variant='shallow')
    elif name=='hqcinn_deep': return HQCINN(n_classes,variant='deep')
    else: raise ValueError(name)

MODEL_NAMES=['hqccnn','peqml','hqcinn_shallow','hqcinn_deep']
print(f'{"Modelo":<20} {"n_classes=2":>14}  {"n_classes=3":>14}')
print('-'*52)
for mn in MODEL_NAMES:
    p2=sum(p.numel() for p in build_model(mn,2).parameters() if p.requires_grad)
    p3=sum(p.numel() for p in build_model(mn,3).parameters() if p.requires_grad)
    print(f'{mn:<20} {p2:>14,}  {p3:>14,}')


Modelo                  n_classes=2     n_classes=3
----------------------------------------------------
hqccnn                       89,686          89,691
peqml                         3,094           3,099
hqcinn_shallow               95,770          95,775
hqcinn_deep                  96,842          96,849
